# Import Libraries

In [1]:
import requests
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries loaded!")

✅ Libraries loaded!


# Define API function

In [2]:
def fetch_nasa_solar_data(lat, lon, start, end, country_name):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,T2M,RH2M,WS2M,PRECTOTCORR",
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON"
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 200:
            data = response.json()
            properties = data["properties"]["parameter"]
            df = pd.DataFrame(properties)
            df.index = pd.to_datetime(df.index, format="%Y%m%d")
            df["country"] = country_name
            return df
    except Exception as e:
        print(f"Error: {e}")
    return None

print("✅ API function ready!")

✅ API function ready!


# Fetch data for all 15 countries

In [3]:
countries_expanded = [
    {"name": "Ethiopia",     "lat": 9.03,   "lon": 38.74},
    {"name": "Benin",        "lat": 9.31,   "lon": 2.32},
    {"name": "Sierra Leone", "lat": 8.46,   "lon": -11.77},
    {"name": "Egypt",        "lat": 26.82,  "lon": 30.80},
    {"name": "South Africa", "lat": -30.56, "lon": 22.94},
    {"name": "India",        "lat": 20.59,  "lon": 78.96},
    {"name": "China",        "lat": 35.86,  "lon": 104.19},
    {"name": "Saudi Arabia", "lat": 23.89,  "lon": 45.08},
    {"name": "Germany",      "lat": 51.16,  "lon": 10.45},
    {"name": "Spain",        "lat": 40.46,  "lon": -3.74},
    {"name": "USA",          "lat": 37.09,  "lon": -95.71},
    {"name": "Brazil",       "lat": -14.24, "lon": -51.93},
    {"name": "Chile",        "lat": -35.67, "lon": -71.54},
    {"name": "Australia",    "lat": -25.27, "lon": 133.77},
    {"name": "UAE",          "lat": 23.42,  "lon": 53.85},
]

all_data_expanded = {}
for c in countries_expanded:
    df = fetch_nasa_solar_data(
        lat=c["lat"], lon=c["lon"],
        start="20230101", end="20231231",
        country_name=c["name"]
    )
    if df is not None:
        all_data_expanded[c["name"]] = df

print(f"✅ Total countries loaded: {len(all_data_expanded)}")

✅ Total countries loaded: 15


# Build ML dataset

In [4]:
coords = {
    "Ethiopia": (9.03, 38.74), "Benin": (9.31, 2.32),
    "Sierra Leone": (8.46, -11.77), "Egypt": (26.82, 30.80),
    "South Africa": (-30.56, 22.94), "India": (20.59, 78.96),
    "China": (35.86, 104.19), "Saudi Arabia": (23.89, 45.08),
    "Germany": (51.16, 10.45), "Spain": (40.46, -3.74),
    "USA": (37.09, -95.71), "Brazil": (-14.24, -51.93),
    "Chile": (-35.67, -71.54), "Australia": (-25.27, 133.77),
    "UAE": (23.42, 53.85)
}

ml_frames = []
for country, df in all_data_expanded.items():
    temp_df = df.copy()
    temp_df["country"] = country
    temp_df["month"] = temp_df.index.month
    temp_df["day_of_year"] = temp_df.index.dayofyear
    temp_df["season"] = temp_df["month"].apply(lambda x:
        "Summer" if x in [6,7,8] else
        "Winter" if x in [12,1,2] else
        "Spring" if x in [3,4,5] else "Autumn")
    temp_df["lat"] = coords[country][0]
    temp_df["lon"] = coords[country][1]
    ml_frames.append(temp_df)

ml_df = pd.concat(ml_frames).reset_index()
ml_df = ml_df.rename(columns={"index": "date"})

# Encode
le_country = LabelEncoder()
le_season = LabelEncoder()
ml_df["country_encoded"] = le_country.fit_transform(ml_df["country"])
ml_df["season_encoded"] = le_season.fit_transform(ml_df["season"])

# Rolling features
ml_df = ml_df.sort_values(["country", "date"])
ml_df["solar_7day_avg"] = ml_df.groupby("country")["ALLSKY_SFC_SW_DWN"].transform(
    lambda x: x.rolling(7, min_periods=1).mean())
ml_df["solar_30day_avg"] = ml_df.groupby("country")["ALLSKY_SFC_SW_DWN"].transform(
    lambda x: x.rolling(30, min_periods=1).mean())
ml_df["clear_sky_ratio"] = (
    ml_df["ALLSKY_SFC_SW_DWN"] / 
    ml_df["CLRSKY_SFC_SW_DWN"].replace(0, 0.001))

print("✅ ML dataset ready!")

✅ ML dataset ready!


# Train and save model

In [5]:
features_v2 = [
    "T2M", "RH2M", "WS2M", "PRECTOTCORR",
    "month", "day_of_year", "country_encoded", "season_encoded",
    "lat", "lon", "solar_7day_avg", "solar_30day_avg",
    "CLRSKY_SFC_SW_DWN", "clear_sky_ratio"
]
target = "ALLSKY_SFC_SW_DWN"

ml_clean_v2 = ml_df[features_v2 + [target]].dropna()
X2 = ml_clean_v2[features_v2]
y2 = ml_clean_v2[target]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42)

print("🤖 Training XGBoost model...")
xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=6,
    learning_rate=0.1, subsample=0.8,
    random_state=42, n_jobs=-1
)
xgb_model.fit(X2_train, y2_train)

y2_pred = xgb_model.predict(X2_test)
r2 = r2_score(y2_test, y2_pred)
mae = mean_absolute_error(y2_test, y2_pred)
print(f"✅ R² Score: {r2:.4f}")
print(f"✅ MAE: {mae:.4f}")

# Save model
os.makedirs("../scripts", exist_ok=True)
joblib.dump(xgb_model, "../scripts/solar_model.pkl")
joblib.dump(le_country, "../scripts/le_country.pkl")
joblib.dump(le_season, "../scripts/le_season.pkl")
print("✅ Model saved to scripts/!")

🤖 Training XGBoost model...
✅ R² Score: 0.9985
✅ MAE: 0.0512
✅ Model saved to scripts/!
